In [1]:
import torch
import datetime as dt
import IPython.display as ipd

from matcha.models.matcha_tts import MatchaTTS
from matcha.hifigan.models import Generator as HiFiGAN
from matcha.hifigan.denoiser import Denoiser
from matcha.hifigan.config import v1
from matcha.utils.utils import intersperse
from matcha.text.symbols import symbols
from matcha.text import text_to_sequence

## Model Testing

In [12]:
CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/logs/multi/checkpoints/last.ckpt"

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab: {vocab_size})...")
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cpu", 
        strict=False,
        n_vocab=vocab_size,
        n_spks=3, 
        spk_emb_dim=64,
        weights_only=False
    )
    m.mel_mean = torch.full((1, 80, 1), -0.842103) 
    m.mel_std = torch.full((1, 80, 1), 1.573202)
    m.eval()
    return m

model_aphasia = load_matcha(CHECKPOINT_PATH, "Multi Model", 198)

print("🚀 Models loaded and ready for testing!")

📦 Loading Multi Model (Vocab: 198)...
🚀 Models loaded and ready for testing!


In [4]:
# --- 1. PRODUCING THE TEXT INPUTS ---
@torch.inference_mode()
def process_text(text: str):
    # 'intersperse' adds blank tokens (0) between phonemes for better rhythm
    seq = text_to_sequence(text, ['english_cleaners2'])[0]
    x = torch.tensor(intersperse(seq, 0), dtype=torch.long)[None] 
    
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long)
    
    # We'll use the symbols to see what it's saying
    x_phones = "".join([symbols[i] for i in x.squeeze(0).tolist() if i < len(symbols)])
    
    return {
        'x_orig': text,
        'x': x,
        'x_lengths': x_lengths,
        'x_phones': x_phones
    }

# --- 2. THE WAVEFORM CONVERTER ---
# Adding a simple denoiser logic since HiFi-GAN can be 'hissy'
def to_waveform(mel, vocoder):
    with torch.no_grad():
        # Ensure mel is [1, 80, T]
        if mel.ndim == 4: mel = mel.squeeze(1)
        
        audio = vocoder(mel).clamp(-1, 1)
        
        # Note: If your local vocoder doesn't have a built-in denoiser, 
        # we skip the 'strength' part to avoid errors, or use a simple squeeze.
        return audio.cpu().squeeze()

# --- 3. THE SYNTHESIS ENGINE ---
@torch.inference_mode()
def synthesise(text, model, n_timesteps=10, temperature=0.667, length_scale=1.0):
    text_processed = process_text(text)
    start_t = dt.datetime.now()
    
    output = model.synthesise(
        text_processed['x'],
        text_processed['x_lengths'],
        n_timesteps=n_timesteps,
        temperature=temperature,
        spks=None,
        length_scale=length_scale
    )
    
    output.update({'start_t': start_t, **text_processed})
    return output

In [5]:
# Use the checkpoint you have in your local folder
HIFIGAN_CHECKPOINT = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/hifigan_T2_v1"

class AttrDict(dict):
    def __init__(self, *args, **kwargs):
        super(AttrDict, self).__init__(*args, **kwargs)
        self.__dict__ = self

def load_vocoder(checkpoint_path):
    h = AttrDict(v1)
    device = torch.device("cpu")
    hifigan = HiFiGAN(h).to(device)
    # We load the 'generator' key from the checkpoint
    state_dict = torch.load(checkpoint_path, map_location=device)
    if 'generator' in state_dict:
        hifigan.load_state_dict(state_dict['generator'])
    else:
        hifigan.load_state_dict(state_dict)
    hifigan.eval()
    hifigan.remove_weight_norm()
    return hifigan

vocoder = load_vocoder(HIFIGAN_CHECKPOINT)
denoiser = Denoiser(vocoder)
print("🎸 Local Vocoder + Denoiser Loaded!")

/rds/general/user/ak8224/home/miniforge3/envs/emojivoice/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Removing weight norm...
🎸 Local Vocoder + Denoiser Loaded!


In [6]:
@torch.inference_mode()
def to_waveform(mel, vocoder, denoiser):
    # Fix the 4D broadcasting error [1, 80, 80, T] -> [1, 80, T]
    if mel.ndim == 4:
        if mel.shape[1] == mel.shape[2] == 80:
            mel = mel[:, 0, :, :] 
        else:
            mel = mel.squeeze(1)

    # Convert to audio
    audio = vocoder(mel).clamp(-1, 1)
    
    # Apply Denoiser (This stops the 'Robotic' metallic sound)
    audio = denoiser(audio.squeeze(0), strength=0.00025).cpu().squeeze()
    
    return audio

In [13]:
def run_synthesis(text, model_to_use, name, id=0):
    print(f"🎬 Testing: {name}")
    
    # 1. Process Text (Intersperse adds the 'breathing room' between phonemes)
    device = model_to_use.device
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    speaker_id = torch.tensor([id], dtype=torch.long, device=device)
    
    # 2. Synthesise Mel
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=10, 
            temperature=0.667, 
            length_scale=1.05,
            spks=speaker_id
        )
        
        # 3. Waveform + Denoise
        audio = to_waveform(output['mel'], vocoder, denoiser)
        
    display(ipd.Audio(audio, rate=22050))

# Run the test
run_synthesis("I uh, am absolutely uhhh. finished with uh... these architecture errors!", model_aphasia, "Aphasia Voice", 0)
run_synthesis("I uh, am absolutely uhhh. finished with uh... these architecture errors!", model_aphasia, "Dementia Voice", 1)
run_synthesis("I uh, am absolutely uhhh. finished with uh... these architecture errors!", model_aphasia, "ASD Voice", 2)

🎬 Testing: Aphasia Voice


🎬 Testing: Dementia Voice


🎬 Testing: ASD Voice


## Script Manipulation

In [14]:
import spacy
import random
import re

# Load the microscopic English NLP model (takes < 1 second)
nlp = spacy.load("en_core_web_sm")

In [15]:
def generate_false_start(word):
    """
    Creates progressive phonetic false starts.
    Example: 'hospital' -> 'ho... hosp... hospital'
    """
    # Strip whitespace and punctuation to get the raw word
    clean_word = re.sub(r'[^\w\s]', '', word)
    
    # If the word is tiny (like "my" or "is"), just stutter the first letter
    if len(clean_word) <= 2:
        return f"{clean_word[0]}... {word}"
        
    stutters = []
    # Weighted random choice: 70% chance of 1 stumble, 30% chance of 2 stumbles
    num_stutters = random.choices([1, 2], weights=[0.7, 0.3])[0]
    
    if num_stutters == 1:
        # Cut the word somewhere in the first half
        cut = random.randint(1, max(1, len(clean_word) - 2))
        stutters.append(f"{clean_word[:cut]}...")
    else:
        # Progressive stutter: cut early, then cut a bit later
        cut1 = max(1, len(clean_word) // 3)
        cut2 = max(cut1 + 1, int(len(clean_word) * 0.75))
        stutters.append(f"{clean_word[:cut1]}...")
        if cut2 < len(clean_word):
            stutters.append(f"{clean_word[:cut2]}...")
            
    # Reattach the trailing whitespace/punctuation from the original word
    return " ".join(stutters) + f" {word}"

def inject_aphasia_cues(text, severity=0.3):
    """Upgraded injector with fillers, repetitions, and false starts."""
    doc = nlp(text)
    fillers = ["um...", "uh,", "uuum...", "..."]
    
    manipulated_words = []
    
    for token in doc:
        is_content_word = token.pos_ in ["NOUN", "VERB", "ADJ", "PROPN"]
        
        if is_content_word and random.random() < severity:
            
            # Now we have 3 clinically accurate dysfluency types!
            stutter_type = random.choices(
                ["filler", "repetition", "false_start"], 
                weights=[0.4, 0.2, 0.4] # 40% filler, 20% repeat word, 40% false start
            )[0]
            
            if stutter_type == "false_start":
                # Inject the progressive stumble and SKIP adding the normal word
                manipulated_words.append(generate_false_start(token.text_with_ws))
                continue 
                
            elif stutter_type == "repetition" and token.i > 0 and doc[token.i - 1].pos_ != "PUNCT":
                prev_word = doc[token.i - 1].text
                manipulated_words.append(f"... {prev_word} ")
                
            elif stutter_type == "filler":
                filler = random.choice(fillers)
                manipulated_words.append(f"{filler} ")
                
        manipulated_words.append(token.text_with_ws)
        
    final_text = "".join(manipulated_words)
    final_text = re.sub(r'\s+([?.!,])', r'\1', final_text)
    return re.sub(r'\s{2,}', ' ', final_text).strip()

In [10]:
# --- LET'S TEST IT ---
clean_script = "Hello doctor, I am feeling a bit dizzy today and my chest hurts."

# Mild Aphasia
print("Mild:  ", inject_aphasia_cues(clean_script, severity=0.15))

# Severe Aphasia
print("Severe:", inject_aphasia_cues(clean_script, severity=0.6))

Mild:   Hello doctor, I am feeling a bit... bit dizzy today and my um... chest hurts.
Severe: Hello do... doctor, I am feelin... feeling a bit dizzy um... today and my chest... hurts.


## Full Inference Pipeline

In [16]:
sentence = "Hello doctor, I am feeling a bit dizzy today and my chest hurts."

manipulated_sentence = inject_aphasia_cues(sentence, severity=0.7)
run_synthesis(manipulated_sentence, model_aphasia, "Aphasia Voice", 0)

🎬 Testing: Aphasia Voice


In [17]:
manipulated_sentence

'Hello... doctor, I am feeling a... a bit... bit dizzy to... toda... today and my che... chest h... hurts.'

## Base VCTK Voice

In [7]:
CHECKPOINT_PATH = r"/rds/general/user/ak8224/home/MedEmoji-TTS/matcha-tts/checkpoints/matcha_vctk.ckpt"

def load_matcha(path, name, vocab_size):
    print(f"📦 Loading {name} (Vocab: {vocab_size})...")
    m = MatchaTTS.load_from_checkpoint(
        path, 
        map_location="cpu", 
        strict=False,
        n_vocab=vocab_size,
        n_spks=109, 
        spk_emb_dim=64,
        weights_only=False
    )
    m.mel_mean = torch.full((1, 80, 1), -6.630575) 
    m.mel_std = torch.full((1, 80, 1), 2.482914)
    m.eval()
    return m

model_vctk = load_matcha(CHECKPOINT_PATH, "Multi Model", 178)

print("🚀 Models loaded and ready for testing!")

📦 Loading Multi Model (Vocab: 178)...


/rds/general/user/ak8224/home/miniforge3/envs/emojivoice/lib/python3.10/site-packages/diffusers/models/lora.py:391: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


🚀 Models loaded and ready for testing!


In [9]:
def run_synthesis(text, model_to_use, name, id=0):
    print(f"🎬 Testing: {name}")
    
    # 1. Process Text (Intersperse adds the 'breathing room' between phonemes)
    device = model_to_use.device
    seq = intersperse(text_to_sequence(text, ['english_cleaners2'])[0], 0)
    x = torch.tensor(seq, dtype=torch.long, device=device)[None]
    x_lengths = torch.tensor([x.shape[-1]], dtype=torch.long, device=device)
    speaker_id = torch.tensor([id], dtype=torch.long, device=device)
    
    # 2. Synthesise Mel
    with torch.no_grad():
        output = model_to_use.synthesise(
            x, x_lengths, 
            n_timesteps=10, 
            temperature=0.667, 
            length_scale=1.05,
            spks=speaker_id
        )
        
        # 3. Waveform + Denoise
        audio = to_waveform(output['mel'], vocoder, denoiser)
        
    display(ipd.Audio(audio, rate=22050))

# Run the test
run_synthesis("This is a test of accent to check if it actually sounds legit", model_vctk, "Indian Accent", 26)

🎬 Testing: Indian Accent
